# Habit & Mood Data — Correlation and Covariance Basics

This notebook builds a small sample DataFrame (like what your Habit + Mood
Dashboard would collect) and walks through **correlation** and **covariance**
step by step, with explanations for each cell.


In [1]:
# Import pandas for the DataFrame, numpy just in case we need it later
import pandas as pd
import numpy as np

pd.set_option('display.width', 100)

## 1. Create a basic sample DataFrame

Six columns, each representing one day's worth of logged data:
- `sleep_hours` — hours slept the night before
- `mood` — self-reported mood, 1-10
- `water_liters` — water intake in liters
- `exercise_minutes` — minutes of exercise that day
- `study_hours` — focused study/work hours
- `screen_time_hours` — hours spent on screens (phone/laptop, non-work)

14 days of made-up but realistic-looking data — enough rows to compute
correlation/covariance meaningfully, but small enough to read at a glance.

In [2]:
# Sample data: 14 days of logs
data = {
    'sleep_hours':       [6.5, 7.0, 5.0, 8.0, 6.0, 7.5, 4.5, 8.5, 6.5, 7.0, 5.5, 8.0, 6.0, 7.5],
    'mood':               [6,   7,   4,   9,   5,   8,   3,   9,   6,   7,   5,   8,   6,   8],
    'water_liters':       [2.0, 2.5, 1.5, 3.0, 2.0, 2.5, 1.0, 3.0, 2.0, 2.5, 1.5, 2.8, 2.0, 2.6],
    'exercise_minutes':   [30,  45,  0,   60,  20,  40,  0,   50,  30,  35,  10,  55,  25,  45],
    'study_hours':        [4,   5,   6,   3,   5,   4,   7,   2,   4,   5,   6,   3,   5,   4],
    'screen_time_hours':  [5,   4,   7,   3,   6,   4,   8,   3,   5,   4,   6,   3,   5,   4],
}

df = pd.DataFrame(data)

# Add a simple date index so it reads like a real log table
df.index = pd.date_range(start='2026-08-01', periods=len(df), freq='D')
df.index.name = 'date'

df

,sleep_hours,mood,water_liters,exercise_minutes,study_hours,screen_time_hours
date,,,,,,
2026-08-01,6.5,6,2.0,30,4,5
2026-08-02,7.0,7,2.5,45,5,4
2026-08-03,5.0,4,1.5,0,6,7
2026-08-04,8.0,9,3.0,60,3,3
2026-08-05,6.0,5,2.0,20,5,6
2026-08-06,7.5,8,2.5,40,4,4
2026-08-07,4.5,3,1.0,0,7,8
2026-08-08,8.5,9,3.0,50,2,3
2026-08-09,6.5,6,2.0,30,4,5


## 2. Compute Correlation

`df.corr()` computes the **Pearson correlation coefficient** between every
pair of numeric columns. The result is always between **-1 and +1**:

- **+1** → perfect positive relationship (as one goes up, the other goes up)
- **-1** → perfect negative relationship (as one goes up, the other goes down)
- **0** → no linear relationship at all

Correlation is **unit-independent** — it doesn't matter that sleep is in hours
and water is in liters, the result is always on the same -1 to +1 scale.
That's what makes it easy to compare across different kinds of columns.

In [3]:
# Correlation matrix across all numeric columns
corr_matrix = df.corr()
corr_matrix

,sleep_hours,mood,water_liters,exercise_minutes,study_hours,screen_time_hours
sleep_hours,1.000000,0.983446,0.979106,0.965300,-0.928173,-0.974214
mood,0.983446,1.000000,0.973895,0.959408,-0.891387,-0.976907
water_liters,0.979106,0.973895,1.000000,0.964718,-0.884942,-0.974611
exercise_minutes,0.965300,0.959408,0.964718,1.000000,-0.872694,-0.970335
study_hours,-0.928173,-0.891387,-0.884942,-0.872694,1.000000,0.879840
screen_time_hours,-0.974214,-0.976907,-0.974611,-0.970335,0.879840,1.000000


In [4]:
# Pull out one specific pair to read more easily,
# e.g. how strongly sleep_hours correlates with mood
sleep_mood_corr = df['sleep_hours'].corr(df['mood'])
print(f"Correlation between sleep and mood: {sleep_mood_corr:.2f}")

# A value close to +1 here means: on days with more sleep, mood tends to be higher.
# This is exactly the kind of number your dashboard's "insights" feature would
# translate into a sentence like: "Your mood tends to rise on days you sleep more."

Correlation between sleep and mood: 0.98


## 3. Compute Covariance

`df.cov()` computes the **covariance** between every pair of numeric columns.
Covariance also tells you the *direction* of a relationship (positive means
they move together, negative means they move opposite), but **unlike
correlation, it is NOT scaled between -1 and 1** — its size depends on the
units and scale of the original columns.

That's why covariance numbers can look huge or tiny and aren't directly
comparable across different column pairs (e.g. covariance between
`sleep_hours` and `mood` isn't directly comparable to covariance between
`study_hours` and `screen_time_hours`, because the columns are on different
scales). Correlation is basically covariance, standardized so you *can*
compare across pairs.

In [5]:
# Covariance matrix across all numeric columns
cov_matrix = df.cov()
cov_matrix

,sleep_hours,mood,water_liters,exercise_minutes,study_hours,screen_time_hours
sleep_hours,1.407967,2.134615,0.694780,21.964286,-1.480769,-1.766484
mood,2.134615,3.346154,1.065385,33.653846,-2.192308,-2.730769
water_liters,0.694780,1.065385,0.357637,11.063187,-0.711538,-0.890659
exercise_minutes,21.964286,33.653846,11.063187,367.719780,-22.500000,-28.434066
study_hours,-1.480769,-2.192308,-0.711538,-22.500000,1.807692,1.807692
screen_time_hours,-1.766484,-2.730769,-0.890659,-28.434066,1.807692,2.335165


In [6]:
# Same single-pair example as before, but with covariance instead
sleep_mood_cov = df['sleep_hours'].cov(df['mood'])
print(f"Covariance between sleep and mood: {sleep_mood_cov:.2f}")

# Notice this number is NOT between -1 and 1 like correlation was.
# It's still positive (same direction as the correlation), but the raw
# magnitude isn't meaningful on its own -- it's only really useful for
# calculating correlation, or for some specific statistical formulas
# (e.g. portfolio variance in finance). For "human-readable insights"
# in your dashboard, correlation is almost always the better number to show.

Covariance between sleep and mood: 2.13


## 4. Quick takeaway

| | Correlation | Covariance |
|---|---|---|
| Range | Always -1 to +1 | Unbounded, depends on units |
| Comparable across column pairs? | Yes | No |
| Easy to turn into a plain-English insight? | Yes ("strong positive relationship") | Not directly |
| What it's good for | Dashboard insights, comparing relationships | Underlying math (e.g. feeds into correlation, used in some statistical models) |

**For the Habit + Mood Dashboard's insights engine, correlation is what you
want to use** — it's the one that translates cleanly into a sentence like
*"You're more productive on days you sleep more."*

One caution worth remembering for the real project: with only ~14-30 days of
a single user's data, these numbers can shift a lot day to day. Treat early
correlations as a rough signal, not a hard fact — and consider only showing
an insight once there's a reasonable amount of data (e.g. 14+ logged days).